# AES-2 Solution v3
No external downloads. Pure sklearn + LightGBM + Ridge ensemble.

In [ ]:
!pip install xgboost textstat -q


In [1]:
import warnings
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Ridge
from scipy.optimize import minimize
from scipy.sparse import hstack as sp_hstack
import xgboost as xgb

try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except Exception:
    GPU_AVAILABLE = False

try:
    import textstat
    HAS_TEXTSTAT = True
except ImportError:
    HAS_TEXTSTAT = False
    print("textstat not found — run: pip install textstat")

warnings.filterwarnings("ignore")
np.random.seed(42)
print("Imports OK")
print(f"GPU available: {GPU_AVAILABLE}")


Imports OK
GPU available: True


In [2]:
DATA_DIR = "./learning-agency-lab-automated-essay-scoring-2/data/learning-agency-lab-automated-essay-scoring-2"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
sub   = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

print(f"Train: {train.shape}  |  Test: {test.shape}")
print("\nScore distribution:")
print(train["score"].value_counts().sort_index().to_string())

Train: (15576, 3)  |  Test: (1731, 2)

Score distribution:
score
1    1124
2    4249
3    5629
4    3563
5     876
6     135


In [3]:
DISCOURSE_MARKERS = [
    "however", "therefore", "furthermore", "moreover", "nevertheless",
    "consequently", "although", "because", "since", "while", "whereas",
    "in conclusion", "in addition", "for example", "for instance",
    "on the other hand", "in contrast", "as a result", "thus", "hence",
    "first", "second", "third", "finally", "in summary",
    "specifically", "notably", "importantly", "additionally",
]

def build_features(df):
    text = df["full_text"]
    f    = pd.DataFrame(index=df.index)

    f["char_count"]          = text.str.len()
    f["word_count"]          = text.str.split().str.len()
    sents                    = text.str.split(r"(?<=[.!?])\s+")
    f["sent_count"]          = sents.str.len()
    f["paragraph_count"]     = text.str.count(r"\n\n") + 1

    word_lens = text.apply(lambda x: [len(w) for w in x.split()] or [0])
    f["avg_word_len"]        = word_lens.apply(np.mean)
    f["max_word_len"]        = word_lens.apply(np.max)
    f["std_word_len"]        = word_lens.apply(np.std)

    sent_lens = sents.apply(lambda ss: [len(s.split()) for s in ss if s.strip()] or [0])
    f["avg_sent_len"]        = sent_lens.apply(np.mean)
    f["std_sent_len"]        = sent_lens.apply(np.std)
    f["max_sent_len"]        = sent_lens.apply(np.max)
    f["min_sent_len"]        = sent_lens.apply(np.min)

    f["unique_words"]        = text.apply(lambda x: len(set(x.lower().split())))
    f["lexical_diversity"]   = f["unique_words"] / (f["word_count"] + 1e-6)
    f["long_word_ratio"]     = text.apply(lambda x: sum(1 for w in x.split() if len(w) >= 7) / (len(x.split()) + 1e-6))
    f["very_long_word_ratio"]= text.apply(lambda x: sum(1 for w in x.split() if len(w) >= 10) / (len(x.split()) + 1e-6))
    f["short_word_ratio"]    = text.apply(lambda x: sum(1 for w in x.split() if len(w) <= 2) / (len(x.split()) + 1e-6))

    f["comma_count"]         = text.str.count(",")
    f["semicolon_count"]     = text.str.count(";")
    f["colon_count"]         = text.str.count(":")
    f["quote_count"]         = text.str.count(r'"')
    f["exclaim_count"]       = text.str.count("!")
    f["question_count"]      = text.str.count(r"\?")
    f["comma_per_sent"]      = f["comma_count"] / (f["sent_count"] + 1e-6)
    f["punct_density"]       = (f["comma_count"] + f["semicolon_count"] + f["colon_count"]) / (f["word_count"] + 1e-6)

    f["cap_ratio"]           = text.apply(lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1e-6))
    f["digit_ratio"]         = text.apply(lambda x: sum(1 for c in x if c.isdigit()) / (len(x) + 1e-6))

    f["discourse_count"]     = text.apply(lambda x: sum(x.lower().count(m) for m in DISCOURSE_MARKERS))
    f["discourse_density"]   = f["discourse_count"] / (f["word_count"] + 1e-6)

    import re
    f["repeat_char_count"]   = text.apply(lambda x: len(re.findall(r"(.)\1{2,}", x)))

    if HAS_TEXTSTAT:
        f["flesch_reading"]  = text.apply(textstat.flesch_reading_ease)
        f["flesch_kincaid"]  = text.apply(textstat.flesch_kincaid_grade)
        f["gunning_fog"]     = text.apply(textstat.gunning_fog)
        f["smog_index"]      = text.apply(textstat.smog_index)
        f["dale_chall"]      = text.apply(textstat.dale_chall_readability_score)
        f["automated_ri"]    = text.apply(textstat.automated_readability_index)

    return f.astype(np.float32)

print("Extracting handcrafted features...")
train_hand = build_features(train)
test_hand  = build_features(test)
print(f"Handcrafted features: {train_hand.shape[1]}")

Extracting handcrafted features...
Handcrafted features: 35


In [4]:
# FIT ON TRAIN ONLY — no leakage
# Note: sklearn TF-IDF/SVD is CPU-based. The tree model below is GPU-enabled through XGBoost.
print("Fitting word TF-IDF (train only)...")
tfidf_word = TfidfVectorizer(max_features=60_000, ngram_range=(1, 3),
                              min_df=2, sublinear_tf=True,
                              strip_accents="unicode", analyzer="word",
                              token_pattern=r"\w{2,}")
svd_word   = TruncatedSVD(n_components=300, n_iter=5, random_state=42)

train_word_raw = tfidf_word.fit_transform(train["full_text"])
train_word_svd = svd_word.fit_transform(train_word_raw).astype(np.float32)
test_word_svd  = svd_word.transform(tfidf_word.transform(test["full_text"])).astype(np.float32)

print("Fitting char TF-IDF (train only)...")
tfidf_char = TfidfVectorizer(max_features=40_000, ngram_range=(3, 5),
                              min_df=5, sublinear_tf=True, analyzer="char_wb")
svd_char   = TruncatedSVD(n_components=150, n_iter=5, random_state=42)

train_char_raw = tfidf_char.fit_transform(train["full_text"])
train_char_svd = svd_char.fit_transform(train_char_raw).astype(np.float32)
test_char_svd  = svd_char.transform(tfidf_char.transform(test["full_text"])).astype(np.float32)

print(f"Word SVD: {train_word_svd.shape}  |  Char SVD: {train_char_svd.shape}")


Fitting word TF-IDF (train only)...
Fitting char TF-IDF (train only)...
Word SVD: (15576, 300)  |  Char SVD: (15576, 150)


In [5]:
test_word_raw = tfidf_word.transform(test["full_text"])
test_char_raw = tfidf_char.transform(test["full_text"])

In [6]:
word_cols = [f"word_{i}" for i in range(300)]
char_cols = [f"char_{i}" for i in range(150)]

X_train_lgb = pd.concat([
    train_hand.reset_index(drop=True),
    pd.DataFrame(train_word_svd, columns=word_cols),
    pd.DataFrame(train_char_svd, columns=char_cols),
], axis=1).astype(np.float32)

X_test_lgb = pd.concat([
    test_hand.reset_index(drop=True),
    pd.DataFrame(test_word_svd, columns=word_cols),
    pd.DataFrame(test_char_svd, columns=char_cols),
], axis=1).astype(np.float32)

# Ridge stays CPU because sklearn Ridge does not use CUDA.
# It is still useful because sparse TF-IDF Ridge often blends well with tree models for essay scoring.
X_train_ridge = sp_hstack([train_word_raw, train_char_raw]).astype(np.float32)
X_test_ridge  = sp_hstack([test_word_raw,  test_char_raw]).astype(np.float32)

y_train = train["score"].values.astype(np.float32)
print(f"XGBoost feature matrix: {X_train_lgb.shape}")
print(f"Ridge feature matrix:   {X_train_ridge.shape}")


XGBoost feature matrix: (15576, 485)
Ridge feature matrix:   (15576, 100000)


In [7]:
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights="quadratic")

def apply_thresholds(y_pred_raw, thresholds):
    return np.clip(np.digitize(y_pred_raw, bins=sorted(thresholds)) + 1, 1, 6).astype(int)

def optimise_thresholds(y_true, y_pred_raw):
    def neg_qwk(t):
        return -cohen_kappa_score(y_true, apply_thresholds(y_pred_raw, t), weights="quadratic")
    best = None
    for x0 in [[1.5,2.5,3.5,4.5,5.5],[1.4,2.4,3.4,4.4,5.2],[1.6,2.6,3.6,4.6,5.6]]:
        r = minimize(neg_qwk, x0=x0, method="Nelder-Mead",
                     options={"maxiter":5000,"xatol":1e-5,"fatol":1e-5})
        if best is None or r.fun < best.fun:
            best = r
    return sorted(best.x.tolist()), -best.fun

print("Helpers defined")

Helpers defined


In [8]:
N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# GPU tree model. XGBoost's modern API uses tree_method='hist' + device='cuda'.
# The fallback handles older environments or notebooks without a working GPU runtime.
def make_xgb_model(use_gpu=True):
    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "learning_rate": 0.02,
        "max_depth": 5,
        "min_child_weight": 2,
        "subsample": 0.85,
        "colsample_bytree": 0.75,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "n_estimators": 3000,
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 100,
    }
    if use_gpu and GPU_AVAILABLE:
        params.update({"tree_method": "hist", "device": "cuda"})
    else:
        params.update({"tree_method": "hist", "device": "cpu"})
    return xgb.XGBRegressor(**params)

oof_lgb    = np.zeros(len(X_train_lgb), dtype=np.float32)   # stores XGBoost OOF predictions
oof_ridge  = np.zeros(len(X_train_lgb), dtype=np.float32)
test_lgb   = np.zeros(len(X_test_lgb), dtype=np.float32)    # stores XGBoost test predictions
test_ridge = np.zeros(len(X_test_lgb), dtype=np.float32)

print(f"{'─'*55}")
print(f"  {N_FOLDS}-Fold Stratified CV  —  GPU XGBoost + Ridge")
print(f"{'─'*55}")
print(f"XGBoost target device: {'cuda' if GPU_AVAILABLE else 'cpu'}")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_lgb, y_train.astype(int))):
    X_tr = X_train_lgb.iloc[tr_idx]
    X_va = X_train_lgb.iloc[val_idx]
    y_tr = y_train[tr_idx]
    y_va = y_train[val_idx]

    # GPU XGBoost, with safe CPU fallback if CUDA/GPU support is unavailable.
    m = make_xgb_model(use_gpu=True)
    try:
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    except Exception as e:
        print(f"  Fold {fold+1}: GPU XGBoost failed, falling back to CPU. Error: {str(e)[:160]}")
        m = make_xgb_model(use_gpu=False)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    oof_lgb[val_idx]  = m.predict(X_va).astype(np.float32)
    test_lgb         += m.predict(X_test_lgb).astype(np.float32) / N_FOLDS

    # CPU Ridge baseline/blend model.
    r = Ridge(alpha=10.0)
    r.fit(X_train_ridge[tr_idx], y_tr)
    oof_ridge[val_idx] = r.predict(X_train_ridge[val_idx]).astype(np.float32)
    test_ridge        += r.predict(X_test_ridge).astype(np.float32) / N_FOLDS

    xgb_q   = qwk(y_train[val_idx], np.clip(np.round(oof_lgb[val_idx]),   1, 6).astype(int))
    ridge_q = qwk(y_train[val_idx], np.clip(np.round(oof_ridge[val_idx]), 1, 6).astype(int))
    print(f"  Fold {fold+1}  |  XGB: {xgb_q:.4f}  |  Ridge: {ridge_q:.4f}  |  trees: {getattr(m, 'best_iteration', None)}")

    del X_tr, X_va, y_tr, y_va, m, r
    gc.collect()

print("\nTraining complete")


───────────────────────────────────────────────────────
  5-Fold Stratified CV  —  GPU XGBoost + Ridge
───────────────────────────────────────────────────────
XGBoost target device: cuda
  Fold 1  |  XGB: 0.7951  |  Ridge: 0.7067  |  trees: 881
  Fold 2  |  XGB: 0.7961  |  Ridge: 0.7098  |  trees: 1353
  Fold 3  |  XGB: 0.7909  |  Ridge: 0.7073  |  trees: 967
  Fold 4  |  XGB: 0.7917  |  Ridge: 0.7057  |  trees: 892
  Fold 5  |  XGB: 0.7892  |  Ridge: 0.7105  |  trees: 1295

Training complete


In [9]:
# Find best XGBoost/Ridge blend weight on OOF
best_w, best_q = 0.5, -1.0
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgb + (1-w) * oof_ridge
    q = qwk(y_train, np.clip(np.round(blend), 1,6).astype(int))
    if q > best_q:
        best_q, best_w = q, w

oof_blend  = best_w * oof_lgb  + (1-best_w) * oof_ridge
test_blend = best_w * test_lgb + (1-best_w) * test_ridge

print(f"XGBoost only QWK: {qwk(y_train, np.clip(np.round(oof_lgb),   1,6).astype(int)):.4f}")
print(f"Ridge only QWK:   {qwk(y_train, np.clip(np.round(oof_ridge), 1,6).astype(int)):.4f}")
print(f"Blend QWK:        {best_q:.4f}  (XGBoost weight = {best_w:.2f})")


XGBoost only QWK: 0.7926
Ridge only QWK:   0.7080
Blend QWK:        0.7926  (XGBoost weight = 1.00)


In [10]:
best_thresholds, best_qwk = optimise_thresholds(y_train, oof_blend)
naive_qwk = qwk(y_train, np.clip(np.round(oof_blend), 1,6).astype(int))

print(f"OOF QWK — naive round:      {naive_qwk:.4f}")
print(f"OOF QWK — threshold tuned:  {best_qwk:.4f}  (+{best_qwk-naive_qwk:.4f})")
print(f"Thresholds: {[round(t,3) for t in best_thresholds]}")

OOF QWK — naive round:      0.7926
OOF QWK — threshold tuned:  0.8112  (+0.0187)
Thresholds: [1.739, 2.548, 3.371, 4.206, 4.911]


In [ ]:
test_preds   = apply_thresholds(test_blend, best_thresholds)
sub["score"] = test_preds
sub.to_csv("predictions_emily.csv", index=False)

print("predictions_emily.csv saved ✓")
print("\nPredicted distribution:")
print(pd.Series(test_preds).value_counts().sort_index().to_string())
print("\nTrain reference:")
print(train["score"].value_counts().sort_index().to_string())

predictions.csv saved ✓

Predicted distribution:
1    116
2    504
3    587
4    376
5    118
6     30

Train reference:
score
1    1124
2    4249
3    5629
4    3563
5     876
6     135


In [ ]:
!aicodinggym mle submit learning-agency-lab-automated-essay-scoring-2 -F predictions.csv